# OSBA 620: Supply Chain Inventory Optimization & Safety Stock Analysis
## Consolidated End-to-End Analytics Project for GitHub Portfolio
**Elise Wallace**

*California State University San Marcos — Masters of Science in Business Analytics*

---

### Executive Project Summary

This project consolidates the end-to-end analytical workflow for supply chain inventory management and safety stock optimization. Operating across multi-site transactional data, our primary objective is to evaluate demand distributions and establish SKU-level safety stock buffers for **Finished Goods (FG)** managed under a **Make-to-Stock (MTS)** replenishment strategy. 

The analytical workflow progresses through four core stages:
1. **Data Cleaning & Anomaly Remediation**: Standardizing dataset attributes, resolving erroneous negative values, filtering non-viable zero-quantity records, correcting multi-year lead time scaling errors, and implementing structured missing value (`NaN`) handling.
2. **Feature Engineering & Subset Aggregation**: Mapping manufacturing locations to geographic country origins and isolating finished goods under make-to-stock control.
3. **Parametric Safety Stock Modeling**: Categorizing SKUs into Normal vs. Negative Binomial demand distributions using the **Coefficient of Variation (CV)** and **Variance-to-Mean Ratio (VMR)**, subsequently deriving Z-score and Reorder Point (ROP) safety stocks across 75%, 90%, and 95% Cycle Service Levels (CSL).
4. **Non-Parametric (Empirical Quantile) Evaluation**: Removing theoretical distribution assumptions by evaluating historical demand percentiles via `scipy.stats.mstats.mquantiles`, producing an empirical safety stock benchmark.

---

# Phase 1: Environment Setup & Raw Data Ingestion

## 1. Importing Libraries & Raw Data Ingestion
We import core data manipulation libraries (`pandas`, `numpy`), statistical distributions from `scipy.stats`, 
and empirical quantile functions from `scipy.stats.mstats`.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.stats import norm
from scipy.stats.mstats import mquantiles

### Loading Datasets
We load the primary transaction log and the corresponding dataset metadata schema.

In [2]:
transaction_data = pd.read_csv("../data/transaction_data.csv")
meta_data = pd.read_csv("../data/meta_data.csv")

meta_data

,feature,description
0,sku_number,Eight character stock keeping unit number
1,inventory_type,Types of inventory including finished good - w...
2,stocking_type,Stockign type policy including make to stock -...
3,lead_time,Total lead time in days
4,unit_price,Unit price US dollar value
5,manufacturing_site,Manufacturing site ID starting with country co...
6,division_code,Four character business division ID
7,transaction_date,Date of Transaction in Year-Month-Day format
8,order_quantity,Order quantity - rounded up for convenience


---

# Phase 2: Data Cleaning & Anomaly Remediation

## 2. Data Cleaning & Header Standardization

### Schema Alignment
Inspecting the raw transaction dataset reveals inconsistent header capitalization and naming conventions. 
We align the transaction headers directly with the metadata schema conventions.

In [3]:
transaction_title = [
    "sku_number", 
    "inventory_type", 
    "stocking_type", 
    "lead_time", 
    "unit_price", 
    "manufacturing_site", 
    "division_code", 
    "transaction_date", 
    "order_quantity"
]

transaction_data.columns = transaction_title
transaction_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 711662 entries, 0 to 711661
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_number          711249 non-null  str    
 1   inventory_type      698248 non-null  str    
 2   stocking_type       697858 non-null  str    
 3   lead_time           711662 non-null  int64  
 4   unit_price          711662 non-null  float64
 5   manufacturing_site  573584 non-null  str    
 6   division_code       661808 non-null  str    
 7   transaction_date    711662 non-null  str    
 8   order_quantity      711662 non-null  int64  
dtypes: float64(1), int64(2), str(6)
memory usage: 48.9 MB


### Numerical Anomaly Identification & Correction

Running descriptive statistics reveals key anomalies:
1. **Negative Order Quantities**: 36,437 records contain negative values due to entry sign inversions. Taking the absolute value (`.abs()`) restores true order volumes.
2. **Zero Order Quantities**: Records with zero quantity represent canceled orders or administrative placeholders. These are omitted (`order_quantity != 0`) to prevent downward bias in demand mean and lead time metrics.
3. **Negative Unit Prices**: Negative price entries represent accounting data-entry errors and are converted using `.abs()`.
4. **Lead Time Outliers**: Extreme lead times exceeding 3 standard deviations above the mean (e.g., up to 2,800 days) reflect historical data-entry scaling errors where lead time was multiplied by 100. Overwriting outlier rows by dividing by 100 restores realistic lead times.

In [4]:
# Step 1: Create a clean working copy
transaction_data_clean = transaction_data.copy()

# Step 2: Take absolute value of order quantities and filter out zero-quantity rows
transaction_data_clean["order_quantity"] = transaction_data_clean["order_quantity"].abs()
transaction_data_clean = transaction_data_clean.query("order_quantity != 0")

# Step 3: Take absolute value of unit prices
transaction_data_clean["unit_price"] = transaction_data_clean["unit_price"].abs()

# Step 4: Correct lead time outliers (> 3 standard deviations above mean)
lt_mean = np.mean(transaction_data_clean["lead_time"])
lt_3std = 3 * np.std(transaction_data_clean["lead_time"])

outlier_condition = (transaction_data_clean["lead_time"] - lt_mean).abs() > lt_3std
transaction_data_clean.loc[outlier_condition, "lead_time"] = (
    transaction_data_clean.loc[outlier_condition, "lead_time"] / 100
)

# Step 5: Verify cleaned numerical distributions
transaction_data_clean.describe()

,lead_time,unit_price,order_quantity
count,707514.000000,707514.000000,707514.000000
mean,25.789379,802.110333,81.047494
std,4.720666,372.236187,40.077371
min,14.000000,32.909604,1.000000
25%,28.000000,982.703171,64.000000
50%,28.000000,996.598691,93.000000
75%,28.000000,1004.506165,109.000000
max,28.000000,1029.941378,198.000000


### Missing Value (NaN) Strategy

Categorical columns (`sku_number`, `inventory_type`, `stocking_type`, `manufacturing_site`, `division_code`) 
contain missing values. To preserve transactional integrity for downstream filtering without dropping 
viable operational records, missing values in categorical features are filled with `"Unknown"`.

In [5]:
categorical_cols = [
    "sku_number", 
    "inventory_type", 
    "stocking_type", 
    "manufacturing_site", 
    "division_code"
]

for col in categorical_cols:
    transaction_data_clean[col] = transaction_data_clean[col].fillna("Unknown")

# Verify zero remaining NaN values across critical fields
transaction_data_clean.isna().sum()

sku_number            0
inventory_type        0
stocking_type         0
lead_time             0
unit_price            0
manufacturing_site    0
division_code         0
transaction_date      0
order_quantity        0
dtype: int64

---

# Phase 3: Feature Engineering & Exploratory Analysis

## 3. Feature Engineering & Subsetting

### Country Code Extraction
Manufacturing site identifiers use standard prefix patterns (`CA` = Canada, `JP` = Japan, `PL` = Poland, `US` = USA). 
Extracting the first two characters (`.str[:2]`) maps transactions to geographic origin.

In [6]:
country_map = {
    "CA": "Canada", 
    "JP": "Japan", 
    "PL": "Poland", 
    "US": "USA"
}

transaction_data_clean["country"] = (
    transaction_data_clean["manufacturing_site"]
    .str[:2]
    .map(country_map)
)

### Isolating Make-to-Stock (MTS) Finished Goods (FG)
Safety stock modeling applies specifically to products with deterministic stocking policies. 
We extract the target subset where `stocking_type == "MTS"` and `inventory_type == "FG"`.

In [7]:
MTS_data = transaction_data_clean.query('stocking_type == "MTS" and inventory_type == "FG"').copy()

# Calculate Total Sales Value per transaction
MTS_data["total_sales_value"] = MTS_data["order_quantity"] * MTS_data["unit_price"]

# Summary Metrics for MTS Portfolio
print(f"Unique SKUs in MTS Portfolio: {MTS_data['sku_number'].nunique()}")
print(f"Unique Manufacturing Sites: {MTS_data['manufacturing_site'].nunique()}")
print(f"Unique Division Codes: {MTS_data['division_code'].nunique()}")

Unique SKUs in MTS Portfolio: 376
Unique Manufacturing Sites: 20
Unique Division Codes: 55


---

# Phase 4: Data Transformation & SKU Aggregation

## 4. SKU-Level Statistical Aggregation

To evaluate SKU demand variability and replenishment lead times, we aggregate order quantities by `sku_number`, 
computing minimum, maximum, mean, median, variance, standard deviation, and average lead time.

In [8]:
transformed_data = (
    transaction_data_clean.query('stocking_type == "MTS"')
    .groupby("sku_number")
    .aggregate({
        "order_quantity": ["min", "max", "mean", "median", "var", "std"],
        "lead_time": "mean"
    })
    .reset_index()
)

# Overwrite MultiIndex column headers
transformed_data.columns = [
    "sku_number",
    "min_order_quantity",
    "max_order_quantity",
    "avg_order_quantity",
    "median_order_quantity",
    "var_order_quantity",
    "std_order_quantity",
    "avg_lead_time"
]

transformed_data.head()

,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time
0,00478A8F,1,166,80.293206,92.0,1575.097842,39.687502,25.714546
1,00874431,1,164,98.524059,100.0,558.627557,23.635303,27.710120
2,018AA89B,1,21,8.262500,7.0,32.120095,5.667459,14.000000
3,01FAD69F,1,23,8.683544,6.0,41.347290,6.430186,14.000000
4,02452E5E,1,87,19.990000,17.0,265.543759,16.295513,19.425000


---

# Phase 5: Demand Distribution Diagnostic & Parametric Safety Stock

## 5. Demand Distribution Analysis & Parametric Modeling

Standard safety stock modeling relies on the **Parametric Normal Distribution** formula:

$$SS = z_{\alpha} \times \sigma_{d} \times \sqrt{LT}$$

However, assuming a bell-curve distribution for highly volatile demand creates severe stockout risk. 
We evaluate two diagnostic metrics per SKU:

1. **Coefficient of Variation (CV)**:
   $$CV = \frac{\sigma_{d}}{\bar{d}}$$
   If $CV \le 0.3$, demand is stable and follows a **Normal Distribution**.

2. **Variance-to-Mean Ratio (VMR)**:
   $$VMR = \frac{\sigma_{d}^2}{\bar{d}}$$
   If $CV > 0.3$ and $VMR \ge 1.2$, demand exhibits heavy-tailed erratic spikes characterized by a **Negative Binomial Distribution**.

In [9]:
# Calculate CV and VMR metrics
transformed_data["cv"] = transformed_data["std_order_quantity"] / transformed_data["avg_order_quantity"]
transformed_data["VMR"] = transformed_data["var_order_quantity"] / transformed_data["avg_order_quantity"]

# Count distribution profiles
normal_skus = transformed_data.query("cv <= 0.3")
negbin_skus = transformed_data.query("cv > 0.3 and VMR >= 1.2")

print(f"SKUs matching Normal Distribution (CV <= 0.3): {len(normal_skus)}")
print(f"SKUs matching Negative Binomial Distribution (CV > 0.3, VMR >= 1.2): {len(negbin_skus)}")

SKUs matching Normal Distribution (CV <= 0.3): 100
SKUs matching Negative Binomial Distribution (CV > 0.3, VMR >= 1.2): 397


### Parametric Normal Safety Stock Calculation

For normally distributed SKUs, Z-scores are computed using `scipy.stats.norm.ppf()`. 
Results are rounded up to integer units without `ceil()` using floor division and boolean comparison:
`(raw_ss // 1 + (raw_ss % 1 > 0))`.

In [10]:
z_75 = norm.ppf(0.75)
z_90 = norm.ppf(0.90)
z_95 = norm.ppf(0.95)

# Calculate raw normal safety stocks
raw_ss_75_norm = z_75 * transformed_data["std_order_quantity"] * np.sqrt(transformed_data["avg_lead_time"])
raw_ss_90_norm = z_90 * transformed_data["std_order_quantity"] * np.sqrt(transformed_data["avg_lead_time"])
raw_ss_95_norm = z_95 * transformed_data["std_order_quantity"] * np.sqrt(transformed_data["avg_lead_time"])

# Round up to whole integers
transformed_data["ss_75_norm"] = (raw_ss_75_norm // 1 + (raw_ss_75_norm % 1 > 0)).astype("int64")
transformed_data["ss_90_norm"] = (raw_ss_90_norm // 1 + (raw_ss_90_norm % 1 > 0)).astype("int64")
transformed_data["ss_95_norm"] = (raw_ss_95_norm // 1 + (raw_ss_95_norm % 1 > 0)).astype("int64")

### Parametric Negative Binomial Safety Stock Calculation

For discrete, skewed demand distributions, Safety Stock is derived from the **Reorder Point (ROP)**:

$$SS = ROP - \mu_{LT}$$

Using the Method of Moments, distribution parameters $p$ and $n$ are calculated across lead time:

$$\mu_{LT} = \bar{d} \times LT, \quad \sigma^2_{LT} = \sigma^2_{d} \times LT$$
$$p = \frac{\mu_{LT}}{\sigma^2_{LT}}, \quad n = \frac{\mu_{LT} \cdot p}{1 - p}$$

The Reorder Point is solved using `scipy.stats.nbinom.ppf(CSL, n, p)`.

In [11]:
transformed_data["mean_during_lt"] = transformed_data["avg_order_quantity"] * transformed_data["avg_lead_time"]
transformed_data["var_during_lt"] = transformed_data["var_order_quantity"] * transformed_data["avg_lead_time"]

transformed_data["p_parameter"] = transformed_data["mean_during_lt"] / transformed_data["var_during_lt"]
transformed_data["n_parameter"] = (
    (transformed_data["mean_during_lt"] * transformed_data["p_parameter"]) / (1 - transformed_data["p_parameter"])
)

# Compute ROP at 75%, 90%, 95% CSL
transformed_data["rop_75"] = stats.nbinom.ppf(0.75, transformed_data["n_parameter"], transformed_data["p_parameter"])
transformed_data["rop_90"] = stats.nbinom.ppf(0.90, transformed_data["n_parameter"], transformed_data["p_parameter"])
transformed_data["rop_95"] = stats.nbinom.ppf(0.95, transformed_data["n_parameter"], transformed_data["p_parameter"])

# Derive Negative Binomial Safety Stock
raw_ss_75_bn = transformed_data["rop_75"] - transformed_data["mean_during_lt"]
raw_ss_90_bn = transformed_data["rop_90"] - transformed_data["mean_during_lt"]
raw_ss_95_bn = transformed_data["rop_95"] - transformed_data["mean_during_lt"]

transformed_data["ss_75_bn"] = (raw_ss_75_bn // 1 + (raw_ss_75_bn % 1 > 0)).astype("int64")
transformed_data["ss_90_bn"] = (raw_ss_90_bn // 1 + (raw_ss_90_bn % 1 > 0)).astype("int64")
transformed_data["ss_95_bn"] = (raw_ss_95_bn // 1 + (raw_ss_95_bn % 1 > 0)).astype("int64")

### Hybrid Safety Stock Policy Selection

We apply `np.where()` to assign Normal Safety Stock to SKUs with $CV \le 0.3$, and Negative Binomial Safety Stock to volatile SKUs ($CV > 0.3$).

In [12]:
transformed_data["ss_75"] = np.where(transformed_data["cv"] <= 0.3, transformed_data["ss_75_norm"], transformed_data["ss_75_bn"])
transformed_data["ss_90"] = np.where(transformed_data["cv"] <= 0.3, transformed_data["ss_90_norm"], transformed_data["ss_90_bn"])
transformed_data["ss_95"] = np.where(transformed_data["cv"] <= 0.3, transformed_data["ss_95_norm"], transformed_data["ss_95_bn"])

transformed_data[["sku_number", "cv", "ss_95_norm", "ss_95_bn", "ss_95"]].head()

,sku_number,cv,ss_95_norm,ss_95_bn,ss_95
0,00478A8F,0.494282,332,342,342
1,00874431,0.239894,205,208,205
2,018AA89B,0.685925,35,37,37
3,01FAD69F,0.740502,40,42,42
4,02452E5E,0.815183,119,125,125


---

# Phase 6: Non-Parametric (Empirical Quantile) Safety Stock Modeling

## 6. Non-Parametric (Empirical Quantile) Method

Parametric models rely on theoretical distribution assumptions. To evaluate safety stock without forcing bell-curve assumptions, we calculate empirical quantiles directly from historical order transactions using `scipy.stats.mstats.mquantiles`.

The non-parametric empirical demand buffer replaces $z_{\alpha} \cdot \sigma_{d}$ with $(\text{Quantile}_{\alpha}(d) - \bar{d})$:

$$\text{Safety Stock} = (\text{Quantile}_{\alpha}(d) - \bar{d}) \times \sqrt{LT}$$

In [13]:
# Extract empirical quantiles per SKU using mquantiles
q75_series = (
    transaction_data_clean.query('stocking_type == "MTS"')
    .groupby("sku_number")["order_quantity"]
    .aggregate(lambda x: mquantiles(x, prob=0.75)[0])
)

q90_series = (
    transaction_data_clean.query('stocking_type == "MTS"')
    .groupby("sku_number")["order_quantity"]
    .aggregate(lambda x: mquantiles(x, prob=0.90)[0])
)

q95_series = (
    transaction_data_clean.query('stocking_type == "MTS"')
    .groupby("sku_number")["order_quantity"]
    .aggregate(lambda x: mquantiles(x, prob=0.95)[0])
)

# Attach empirical quantiles to transformed_data
transformed_data["q75_order_quantity"] = transformed_data["sku_number"].map(q75_series)
transformed_data["q90_order_quantity"] = transformed_data["sku_number"].map(q90_series)
transformed_data["q95_order_quantity"] = transformed_data["sku_number"].map(q95_series)

# Calculate non-parametric empirical safety stock
raw_ss_np_75 = (transformed_data["q75_order_quantity"] - transformed_data["avg_order_quantity"]) * np.sqrt(transformed_data["avg_lead_time"])
raw_ss_np_90 = (transformed_data["q90_order_quantity"] - transformed_data["avg_order_quantity"]) * np.sqrt(transformed_data["avg_lead_time"])
raw_ss_np_95 = (transformed_data["q95_order_quantity"] - transformed_data["avg_order_quantity"]) * np.sqrt(transformed_data["avg_lead_time"])

transformed_data["ss_np_75"] = (raw_ss_np_75 // 1 + (raw_ss_np_75 % 1 > 0)).clip(lower=0).astype("int64")
transformed_data["ss_np_90"] = (raw_ss_np_90 // 1 + (raw_ss_np_90 % 1 > 0)).clip(lower=0).astype("int64")
transformed_data["ss_np_95"] = (raw_ss_np_95 // 1 + (raw_ss_np_95 % 1 > 0)).clip(lower=0).astype("int64")

---

# Phase 7: Comparative Inventory Policy Analysis & Strategic Insights

## 7. Comparative Inventory Policy Analysis (95% Service Level)

We construct a comparative summary dataframe isolating the 95% service level outputs across:
1. `ss_95_norm`: Parametric Normal Model
2. `ss_95_bn`: Parametric Negative Binomial Model
3. `ss_95`: Hybrid Distribution-Matched Model
4. `ss_np_95`: Non-Parametric Empirical Quantile Model

In [14]:
selected_cols = ["sku_number", "ss_95_norm", "ss_95_bn", "ss_95", "ss_np_95"]
data_95_ss = transformed_data[selected_cols].copy()

# Summary Statistics across Policy Methods
data_95_ss.describe()

,ss_95_norm,ss_95_bn,ss_95,ss_np_95
count,497.000000,497.000000,497.000000,497.000000
mean,119.997988,123.674044,123.146881,113.981891
std,96.063036,97.887579,97.463251,71.136013
min,29.000000,30.000000,30.000000,32.000000
25%,41.000000,44.000000,44.000000,49.000000
50%,66.000000,69.000000,69.000000,76.000000
75%,176.000000,178.000000,176.000000,178.000000
max,353.000000,370.000000,370.000000,323.000000


### SKU Risk & Safety Stock Leaderboards

Comparing top and bottom SKU safety stock requirements across methods highlights distribution sensitivity:

In [15]:
# Top 5 SKUs by Hybrid Safety Stock
print("--- Top 5 SKUs (Hybrid Distribution-Matched Model) ---")
display(data_95_ss.sort_values("ss_95", ascending=False).head(5))

# Top 5 SKUs by Non-Parametric Model
print("--- Top 5 SKUs (Non-Parametric Empirical Quantile Model) ---")
display(data_95_ss.sort_values("ss_np_95", ascending=False).head(5))

--- Top 5 SKUs (Hybrid Distribution-Matched Model) ---


,sku_number,ss_95_norm,ss_95_bn,ss_95,ss_np_95
342,AC3ADABF,352,370,370,323
20,089FA02B,353,366,366,263
471,F0610752,351,362,362,261
489,FDDDA064,349,360,360,257
156,56D83D7A,348,359,359,258


--- Top 5 SKUs (Non-Parametric Empirical Quantile Model) ---


,sku_number,ss_95_norm,ss_95_bn,ss_95,ss_np_95
342,AC3ADABF,352,370,370,323
166,5978F66E,345,358,358,278
62,248D68DF,330,340,340,266
20,089FA02B,353,366,366,263
471,F0610752,351,362,362,261


In [18]:
# Bottom 5 SKUs by Hybrid Safety Stock
print("--- Bottom 5 SKUs (Hybrid Distribution-Matched Model) ---")
display(data_95_ss.sort_values("ss_95", ascending=True).head(5))

# Bottom 5 SKUs by Non-Parametric Model
print("--- Bottom 5 SKUs (Non-Parametric Empirical Quantile Model) ---")
display(data_95_ss.sort_values("ss_np_95", ascending=True).head(5))

--- Bottom 5 SKUs (Hybrid Distribution-Matched Model) ---


,sku_number,ss_95_norm,ss_95_bn,ss_95,ss_np_95
163,58F77D5E,29,30,30,32
61,243ECBFB,31,33,33,35
495,FF9E44A8,32,33,33,37
49,1F900448,33,34,34,35
384,C6D49812,34,35,35,42


--- Bottom 5 SKUs (Non-Parametric Empirical Quantile Model) ---


,sku_number,ss_95_norm,ss_95_bn,ss_95,ss_np_95
163,58F77D5E,29,30,30,32
61,243ECBFB,31,33,33,35
49,1F900448,33,34,34,35
445,E0C722AA,37,38,38,36
380,C565BADC,40,42,42,36


In [20]:
# Avg SS by all models
data_95_ss.describe()

,ss_95_norm,ss_95_bn,ss_95,ss_np_95
count,497.000000,497.000000,497.000000,497.000000
mean,119.997988,123.674044,123.146881,113.981891
std,96.063036,97.887579,97.463251,71.136013
min,29.000000,30.000000,30.000000,32.000000
25%,41.000000,44.000000,44.000000,49.000000
50%,66.000000,69.000000,69.000000,76.000000
75%,176.000000,178.000000,176.000000,178.000000
max,353.000000,370.000000,370.000000,323.000000


### Key Strategic Takeaways & Inventory Impact

1. **Elimination of Distribution Assumption Bias**:
   Parametric models force customer demand into symmetrical bell curves or theoretical count distributions. 
   When actual SKU demand exhibits long tail behavior interspersed with zero-demand periods, standard deviations over-inflate tail risk. 
   The non-parametric empirical quantile method removes theoretical distortion by grounding buffers directly in historical customer order behavior.

2. **Financial Capital Efficiency (~114 vs. ~123 Average Units)**:
   Adopting the empirical quantile method reduces average safety stock across all SKUs to **~114 units**, compared to **~123 units** under parametric hybrid models and **~124 units** under pure negative binomial modeling. 
   This **~7.3% reduction in holding inventory** directly frees up working capital and lowers storage costs without compromising the target 95% Cycle Service Level.

3. **SKU Ranking Shifts**:
   - **High-Risk Items**: SKU `AC3ADABF` tops safety stock requirements under both Negative Binomial and Non-Parametric models, confirming genuine structural demand volatility. However, the non-parametric method significantly lowers the absolute buffer size, preventing severe over-stocking.
   - **Low-Risk Items**: SKU `58F77D5E` maintains the lowest safety stock across all models, but the non-parametric method assigns a slightly higher buffer than parametric models. This reflects discrete order batching (pack sizes) that smooth parametric curves under-represent.

---

## References & Attribution
- CASRAI. (n.d.). *Negative binomial regression overdispersion*. CASRAI Guides.
- Oracle. (n.d.). *NetSuite help center: Inventory management guidelines*. Oracle Help Center.
- Pandas Development Team. (n.d.). *Pandas.DataFrame.aggregate documentation*.
- SciPy Development Team. (n.d.). *scipy.stats.mstats.mquantiles documentation*.
- Zaiontz, C. (n.d.). *Method of moments: Negative binomial distribution*. Real Statistics Using Excel.